<a href="https://colab.research.google.com/github/synatkeamsk/rbc-morphology/blob/main/notebooks/colab_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Wed Nov 11 04:16:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Clone your repo and enter the folder
!git clone https://github.com/synatkeamsk/rbc-morphology.git
%cd rbc-morphology

# Install remaining libraries
!pip install -q timm albumentations opencv-python-headless onnx onnxruntime

Cloning into 'rbc-morphology'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 78 (delta 26), reused 55 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 438.48 KiB | 1.78 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/rbc-morphology


In [3]:
!python src/model.py

In [4]:
import os
import cv2
import numpy as np
import pandas as pd

# Create dummy sample RBC patches
os.makedirs("data/raw/synthetic", exist_ok=True)
os.makedirs("data/splits", exist_ok=True)

records = []
classes = ["normal", "drepanocyte", "dacrocyte", "schistocyte", "echinocyte", "elliptocyte", "spherocyte", "codocyte"]

for i in range(160):
    label_idx = i % len(classes)
    cls_name = classes[label_idx]

    # Create a 224x224 RGB image (simulating a stained RBC crop)
    img = np.random.randint(180, 240, (224, 224, 3), dtype=np.uint8)
    cv2.circle(img, (112, 112), 60, (120, 50, 160), -1) # Cell body
    cv2.circle(img, (112, 112), 25, (220, 200, 220), -1) # Central pallor

    path = f"data/raw/synthetic/cell_{i}_{cls_name}.png"
    cv2.imwrite(path, img)
    records.append({"image_path": path, "label": label_idx})

df = pd.DataFrame(records)
# Split 80% train, 20% val
train_df = df.sample(frac=0.8, random_state=42)
val_df = df.drop(train_df.index)

train_df.to_csv("data/splits/train.csv", index=False)
val_df.to_csv("data/splits/val.csv", index=False)

print(f"Created {len(train_df)} train samples and {len(val_df)} validation samples.")

Created 128 train samples and 32 validation samples.


In [5]:
%%bash
# Add project root to sys.path in src/train.py
python3 -c "
import sys
content = open('src/train.py').read()
if 'sys.path.append' not in content:
    header = 'import sys\nfrom pathlib import Path\nsys.path.append(str(Path(__file__).resolve().parents[1]))\n\n'
    open('src/train.py', 'w').write(header + content)
"

# Add project root to sys.path in src/infer.py
python3 -c "
import sys
content = open('src/infer.py').read()
if 'sys.path.append' not in content:
    header = 'import sys\nfrom pathlib import Path\nsys.path.append(str(Path(__file__).resolve().parents[1]))\n\n'
    open('src/infer.py', 'w').write(header + content)
"

In [6]:
import re
from pathlib import Path

# 1. Update backbone name to 'convnext_nano' in all files
for file in ["src/config.py", "src/model.py", "src/train.py", "src/infer.py"]:
    p = Path(file)
    if p.exists():
        content = p.read_text().replace("convnext_nano.in12k_ft_in1k_224", "convnext_nano")
        p.write_text(content)

# 2. Update default workers from 4 to 2 (optimized for Colab's 2 CPU cores)
for file, old_val, new_val in [
    ("src/config.py", "num_workers: int = 4", "num_workers: int = 2"),
    ("src/train.py", "default=4", "default=2"),
]:
    p = Path(file)
    if p.exists():
        p.write_text(p.read_text().replace(old_val, new_val))

# 3. Replace deprecated ShiftScaleRotate with modern Affine in src/data.py
p = Path("src/data.py")
if p.exists():
    content = p.read_text()
    content = re.sub(
        r"A\.ShiftScaleRotate\([^)]*p=0\.7\s*\)",
        "A.Affine(scale=(0.9, 1.1), translate_percent=(-0.06, 0.06), rotate=(-180, 180), border_mode=cv2.BORDER_REFLECT, p=0.7)",
        content,
        flags=re.DOTALL
    )
    p.write_text(content)

print("All fixes applied cleanly!")

All fixes applied cleanly!


In [7]:
!python -m src.train \
    --train_csv data/splits/train.csv \
    --val_csv data/splits/val.csv \
    --num_classes 8 \
    --backbone convnext_nano \
    --epochs 5 \
    --batch_size 16 \
    --num_workers 2 \
    --device cuda


model.safetensors: downloading bytes:  95% 59.5M/62.4M [00:01<00:00, 89.1MB/s, 4.40MB/s  ]
model.safetensors: downloading bytes: 100% 59.5M/59.5M [00:01<00:00, 45.6MB/s, 5.72MB/s  ]
model.safetensors: reconstructing file: 100% 62.4M/62.4M [00:01<00:00, 47.9MB/s, 6.02MB/s  ]
/content/rbc-morphology/src/train.py:109: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Starting SmearScope training loop...
Training:   0% 0/8 [00:00<?, ?it/s]/content/rbc-morphology/src/train.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validating:   0% 0/2 [00:00<?, ?it/s]/content/rbc-morphology/src/train.py:71: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch [01/05] Train Loss: 0.0208 | Train Acc: 0.172 | Va

In [8]:
!pip install -q onnxscript

In [ ]:
!python -m src.infer \
    --export \
    --checkpoint checkpoints/best_smearscope.pt \
    --onnx_path checkpoints/best_smearscope.onnx \
    --num_classes 8

Loading checkpoint: checkpoints/best_smearscope.pt
Exporting ONNX: checkpoints/best_smearscope.onnx
/content/rbc-morphology/src/infer.py:102: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
[torch.onnx] Obtain model graph for `SmearScopeClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SmearScopeClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


In [ ]:
from src.infer import SmearPredictor

# Target classes
classes = ["normal", "drepanocyte", "dacrocyte", "schistocyte", "echinocyte", "elliptocyte", "spherocyte", "codocyte"]

# Initialize ONNX inference engine
predictor = SmearPredictor(
    model_path="checkpoints/best_smearscope.onnx",
    classes=classes,
    backend="onnx"
)

# Test on one of synthetic RBC images
sample_image = "data/raw/synthetic/cell_0_normal.png"
result = predictor.predict(sample_image)

print(f"Predicted Class: {result['prediction']}")
print(f"Confidence:      {result['confidence'] * 100:.2f}%\n")
print("Class Probabilities:")
for cls, prob in result["probabilities"].items():
    print(f"  - {cls:<12}: {prob * 100:6.2f}%")

In [ ]:
# Clean previous runs, clone fresh, and enter directory
!rm -rf rbc-morphology
!git clone https://github.com/synatkeamsk/rbc-morphology.git
%cd rbc-morphology

# Verify NVIDIA T4 GPU is assigned
!nvidia-smi